# Vista v2 — o Vista completo no simulador

Continua do treino de 02/11 (`treino_vista.ipynb`). O que mudou, decidido com a equipe em 09/11 (ADR 008):

| | Antes (v1) | Agora (v2) |
|---|---|---|
| **Cruzamento** | a rede de faixa tentava dirigir lá dentro — e se perdia (4/10 rotas) | a curva sai do **mapa oficial** e é seguida por **odometria** (encoders + giroscópio); a rede só dirige na faixa |
| **Placas** | manobra vinha da verdade | **detector (YOLO) no laço**: cada placa vista é atribuída à aproximação do mapa que a explica; `rules.decide` escolhe |
| **Semáforo** | verdade | **detector**: larga com verde em 2 leituras seguidas |
| **Localização** | — | grafo do mapa (o Vista sabe em que rua está porque ele mesmo decidiu as conversões) + pose amarrada à faixa pela rede |
| **Giroscópio** | viés calibrado no vermelho | + **escala calibrada a cada curva** (a MPU6050 só garante ±3 %) |
| **Velocidade** | 0,5 m/s | dados com **0,35–0,80 m/s**; avaliação a 0,5 e depois subindo |

Nada de verdade de campo no Vista: imagem, telemetria, mapa e calibração da câmera. A verdade só **mede**.

**Como usar:** GPU (L4) + `Executar tudo`. Precisa da pasta do Drive com os modelos de 02/11 (`MODELOS_DE`). Os resultados vão para uma pasta nova no Drive.

## 1. Parâmetros

In [ ]:
REPO = "https://github.com/Gustavo-dev04/C.A.FATEC.R.B.git"
BRANCH = "claude/autonomous-car-repo-data-lzwn7t"     # depois do merge: "main"

# Pasta do Drive com os modelos do treino anterior (faixa_final.* ou faixa.onnx + faixa_melhor.pt, placas.onnx)
MODELOS_DE = "/content/drive/MyDrive/vista/modelos/2026-11-02_1929"

# --- avaliação no simulador (sementes NUNCA vistas no treino) --------------
SEMENTES_AVALIACAO = list(range(100000, 100020))
VELOCIDADES_TESTE = [0.5, 0.6, 0.7, 0.8]       # m/s — só depois de completar a 0,5

# --- ajuste da rede de faixa com dados em várias velocidades ----------------
AJUSTAR_FAIXA = True
EPISODIOS = 150                  # dados novos (o oráculo sorteia a velocidade em cada um)
SEMENTE_INICIAL = 200000         # bloco novo de sementes (os de 02/11 eram 0..149)
VELOCIDADES_DADOS = "0.35:0.80"
EPOCAS_AJUSTE = 8
LR_AJUSTE = 2e-4
LOTE_FAIXA = 256

# --- DAgger com o Vista: ele dirige, o oráculo socorre -----------------------
RODADAS_DAGGER = 2
EPISODIOS_DAGGER = 40
EPOCAS_DAGGER = 6
LR_DAGGER = 2e-4

# --- detector ---------------------------------------------------------------
TREINAR_DETECTOR = False         # True: retreina o YOLO com os dados novos (≈1 h a mais)
EPOCAS_YOLO = 60
IMG_YOLO = 416

# --- experimento (opcional): outra montagem da câmera ------------------------
# None = config/camera/extrinsics.yaml (15°). Com outro valor, os dados são
# gerados e a rede treinada para ESSE pitch (ver seção 9).
PITCH_GRAUS = None

PASTA_DRIVE = "/content/drive/MyDrive/vista"

## 2. Ambiente

In [ ]:
import os, subprocess, sys, json, time, shutil
from pathlib import Path

REPO_DIR = Path("/content/vista")
if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("Clone público falhou — tentando com o GITHUB_TOKEN dos Segredos do Colab...")
        from google.colab import userdata
        url = REPO.replace("https://", f"https://x-access-token:{userdata.get('GITHUB_TOKEN')}@")
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("não consegui clonar o repositório (token sem acesso?)")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", COMMIT)

In [ ]:
%pip install -q onnx onnxruntime onnxscript ultralytics
import torch, cv2, numpy as np
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sem GPU")
WORKERS = max(1, (os.cpu_count() or 2) - 1)
print("CPUs:", os.cpu_count(), "→ avaliação com", WORKERS, "processos")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAIDA = Path(PASTA_DRIVE) / "modelos" / (time.strftime("%Y-%m-%d_%H%M") + "_v2")
SAIDA.mkdir(parents=True, exist_ok=True)
print("tudo que importa vai para:", SAIDA)

## 3. Modelos de 02/11

Pega a rede de faixa (de preferência a escolhida pelo DAgger, `faixa_final`) e o detector `placas.onnx`.
O `.pt` é usado para avaliar na GPU (rápido) e para continuar o treino; o `.onnx` é o que vai para a Jetson.

In [ ]:
ant = Path(MODELOS_DE)
assert ant.exists(), f"não achei {ant} — confira MODELOS_DE"
def primeiro(*nomes):
    for n in nomes:
        if (ant / n).exists():
            return ant / n
    return None

FAIXA_PT = primeiro("faixa_final.pt", "faixa_melhor.pt")
FAIXA_ONNX = primeiro("faixa_final.onnx", "faixa.onnx")
PLACAS_ONNX = primeiro("placas.onnx")
assert FAIXA_PT and FAIXA_ONNX and PLACAS_ONNX, "faltam modelos em MODELOS_DE"
for f in (FAIXA_PT, FAIXA_ONNX, FAIXA_ONNX.with_suffix(".json"), PLACAS_ONNX):
    shutil.copy(f, SAIDA / ("anterior_" + f.name))
print("faixa:", FAIXA_PT.name, "+", FAIXA_ONNX.name, "| detector:", PLACAS_ONNX.name)

## 4. Antes × agora, com os MESMOS modelos

Mesmas rotas, quatro configurações:

1. **teto do Vista** — rede de faixa e detector perfeitos (a verdade no lugar deles). Mede o resto: mapa, odometria, curva, leitura de placa, regras.
2. **Vista com a rede de faixa** — detector ainda perfeito (isola a rede).
3. **Vista completo** — rede de faixa + YOLO. É o que vai para o carro.
4. **antes (v1)** — a rede dirigindo também dentro do cruzamento, manobra da verdade (o 4/10 de 02/11).

Cada rota roda num processo; leva alguns minutos por configuração.

In [ ]:
from treino.avaliacao import rodar_vista, rodar, PreditorOnnx

def mostra(nome, r):
    print(f"{nome:28s} {r['completas']:2d}/{r['rotas']} completas | progresso {r['progresso_medio']*100:3.0f}% | "
          f"manobras {r['manobras_certas']}/{r['decisoes']} | placas {r['placas_certas']}/{r['decisoes']} | "
          f"reserva {r['reservas']} | largou antes do verde {r['largadas_antes_do_verde']} | {r['motivos']}")

aval = {}
aval["teto"] = rodar_vista(SEMENTES_AVALIACAO, None, "verdade", workers=WORKERS, log=lambda s: None)
mostra("teto do Vista", aval["teto"])
aval["rede_faixa"] = rodar_vista(SEMENTES_AVALIACAO, FAIXA_PT, "verdade", workers=WORKERS, log=lambda s: None)
mostra("Vista c/ rede de faixa", aval["rede_faixa"])
aval["completo"] = rodar_vista(SEMENTES_AVALIACAO, FAIXA_PT, PLACAS_ONNX, workers=WORKERS, log=lambda s: None)
mostra("Vista completo", aval["completo"])
(SAIDA / "avaliacao_v2_modelos_anteriores.json").write_text(json.dumps(aval, indent=2, ensure_ascii=False))

In [ ]:
# antes (v1): para comparar — a rede dirige em todo lugar (sequencial, mais lento)
from treino.faixa import carregar, PreditorTorch
modelo, tam = carregar(FAIXA_PT, device="cuda" if torch.cuda.is_available() else "cpu")
antes = rodar(PreditorTorch(modelo, tam), SEMENTES_AVALIACAO[:10], filtro=True, log=lambda s: None)
print(f"antes (v1), 10 rotas: {antes['completas']}/{antes['rotas']} completas | {antes['motivos']}")

In [ ]:
# Onde o Vista completo falhou: a placa lida × a certa, e o que o carro fez
for r in aval["completo"]["por_rota"]:
    if r["motivo"] != "rota_completa":
        erradas = [l for l in r["leituras"] if l["manobra"] != l["esperada"] or l["placa"] != l["certa"]]
        print(r["semente"], r["motivo"], f"progresso {r['progresso']*100:.0f}%", r.get("falha"), erradas[:2])

## 5. Dados novos: várias velocidades

O oráculo sorteia a velocidade de cruzeiro em cada episódio (`VELOCIDADES_DADOS`) e a de conversão em 0,24–0,34 m/s. Sementes num bloco novo (200000+): a validação continua separada por bloco.

In [ ]:
DADOS = Path("/content/dataset_v2")
extra = f"--pitch-graus {PITCH_GRAUS}" if PITCH_GRAUS is not None else ""
if AJUSTAR_FAIXA or TREINAR_DETECTOR or RODADAS_DAGGER:
    if not (DADOS / "meta.json").exists():
        est = EPISODIOS * 30 / max(os.cpu_count(), 1) / 60
        print(f"gerando {EPISODIOS} episódios (~{est:.0f} min)...")
        !python -m tools.gerar_dataset --out {DADOS} --episodios {EPISODIOS} --semente-inicial {SEMENTE_INICIAL} --velocidades {VELOCIDADES_DADOS} {extra}
    meta = json.loads((DADOS / "meta.json").read_text())
    print(f"{meta['episodios']} episódios, {meta['imagens']} imagens, velocidades {meta['velocidades']}")

## 6. Ajuste da rede de faixa

Parte do modelo de 02/11 (não do zero) e ajusta com os dados novos. Com `PITCH_GRAUS` diferente de 15°, é praticamente outra câmera: aumente `EPOCAS_AJUSTE`.

In [ ]:
from treino.faixa import treinar, exportar_onnx
dev = "cuda" if torch.cuda.is_available() else "cpu"
atual_pt, atual_onnx = FAIXA_PT, FAIXA_ONNX
if AJUSTAR_FAIXA:
    pasta = SAIDA / "ajuste"
    pasta.mkdir(exist_ok=True)
    res = treinar(DADOS, pasta, epocas=EPOCAS_AJUSTE, lote=LOTE_FAIXA, lr=LR_AJUSTE,
                  workers=min(os.cpu_count(), 12), inicial=FAIXA_PT)
    modelo, tam = carregar(res["melhor"], device=dev)
    print(exportar_onnx(modelo, tam, pasta / "faixa.onnx"))
    atual_pt, atual_onnx = res["melhor"], pasta / "faixa.onnx"
    melhor = min(res["historico"], key=lambda m: m["perda"])
    print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in melhor.items()}, indent=1))
    aval["ajuste"] = rodar_vista(SEMENTES_AVALIACAO, atual_pt, PLACAS_ONNX, workers=WORKERS,
                                 pitch_graus=PITCH_GRAUS, log=lambda s: None)
    mostra("Vista completo (ajustada)", aval["ajuste"])

## 7. DAgger com o Vista

Agora quem dirige é o **Vista** (rede na faixa, mapa + odometria no cruzamento); o oráculo assume quando o carro passa de 6 cm ou 25° e devolve quando estabiliza. As imagens dessas situações entram no treino com o rótulo exato. A cada rodada a rede é ajustada com todos os dados e o Vista completo é avaliado de novo; fica a melhor rodada.

In [ ]:
historico = [{"rodada": 0, "pt": str(atual_pt), "onnx": str(atual_onnx),
              **{k: aval.get("ajuste", aval["completo"])[k] for k in ("completas", "rotas", "progresso_medio")}}]
if RODADAS_DAGGER and not (DADOS / "meta.json").exists():
    raise SystemExit("o DAgger precisa da pasta de dados (seção 5)")
for r in range(1, RODADAS_DAGGER + 1):
    pasta_r = SAIDA / f"dagger_{r}"
    pasta_r.mkdir(exist_ok=True)
    print(f"\n=== rodada {r}: o Vista dirige {EPISODIOS_DAGGER} rotas, o oráculo socorre ===")
    !python -m tools.gerar_dataset --out {DADOS} --episodios {EPISODIOS_DAGGER} --semente-inicial {600000 + 1000 * r} --vistas-extras 0 --piloto-onnx {atual_onnx} --velocidades {VELOCIDADES_DADOS} {extra}
    res_r = treinar(DADOS, pasta_r, epocas=EPOCAS_DAGGER, lote=LOTE_FAIXA, lr=LR_DAGGER,
                    workers=min(os.cpu_count(), 12), inicial=atual_pt)
    modelo, tam = carregar(res_r["melhor"], device=dev)
    exportar_onnx(modelo, tam, pasta_r / "faixa.onnx")
    atual_pt, atual_onnx = res_r["melhor"], pasta_r / "faixa.onnx"
    av = rodar_vista(SEMENTES_AVALIACAO, atual_pt, PLACAS_ONNX, workers=WORKERS, pitch_graus=PITCH_GRAUS,
                     log=lambda s: None)
    (pasta_r / "avaliacao.json").write_text(json.dumps(av, indent=2, ensure_ascii=False))
    mostra(f"Vista completo (DAgger {r})", av)
    historico.append({"rodada": r, "pt": str(atual_pt), "onnx": str(atual_onnx),
                      **{k: av[k] for k in ("completas", "rotas", "progresso_medio")}})

melhor_r = max(historico, key=lambda x: (x["completas"], x["progresso_medio"]))
shutil.copy(melhor_r["onnx"], SAIDA / "faixa_final.onnx")
shutil.copy(Path(melhor_r["onnx"]).with_suffix(".json"), SAIDA / "faixa_final.json")
shutil.copy(melhor_r["pt"], SAIDA / "faixa_final.pt")
(SAIDA / "dagger.json").write_text(json.dumps(historico, indent=2, ensure_ascii=False))
print("\nrodada | completas | progresso")
for h_ in historico:
    print(f"  {h_['rodada']}    | {h_['completas']}/{h_['rotas']}     | {h_['progresso_medio']*100:.0f}%")
print(f"escolhida: rodada {melhor_r['rodada']} → faixa_final.onnx")
FAIXA_FINAL_PT = SAIDA / "faixa_final.pt"

## 8. Detector (opcional) e subir a velocidade

Primeiro o objetivo é **completar a 0,5 m/s**. Com isso feito, o mesmo Vista é avaliado mais rápido — o que limita é o carro (conversões a 0,28 m/s, rampa de 1 m/s²) e o quanto a rede aguenta de borrão e de atraso.

In [ ]:
PLACAS_FINAL = PLACAS_ONNX
if TREINAR_DETECTOR:
    from treino.deteccao import montar_yolo
    from ultralytics import YOLO
    resumo = montar_yolo(DADOS, Path("/content/yolo_v2"), val_frac=0.1)
    det = YOLO(str(Path(MODELOS_DE) / "placas.pt") if (Path(MODELOS_DE) / "placas.pt").exists() else "yolo11n.pt")
    det.train(data=resumo["data_yaml"], epochs=EPOCAS_YOLO, imgsz=IMG_YOLO, batch=64, patience=20,
              fliplr=0.0, degrees=0.0, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, mosaic=1.0,
              workers=min(os.cpu_count(), 12), project="/content/yolo_runs", name="placas_v2", exist_ok=True)
    best = Path("/content/yolo_runs/placas_v2/weights/best.pt")
    det = YOLO(str(best))
    m = det.val(data=resumo["data_yaml"], imgsz=IMG_YOLO, split="val")
    print(f"mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f}")
    shutil.copy(det.export(format="onnx", imgsz=IMG_YOLO, opset=17, simplify=True), SAIDA / "placas.onnx")
    shutil.copy(best, SAIDA / "placas.pt")
    PLACAS_FINAL = SAIDA / "placas.onnx"
else:
    shutil.copy(PLACAS_ONNX, SAIDA / "placas.onnx")

velocidades = {}
for v in VELOCIDADES_TESTE:
    velocidades[v] = rodar_vista(SEMENTES_AVALIACAO, FAIXA_FINAL_PT, PLACAS_FINAL, v_cruzeiro=v,
                                 workers=WORKERS, pitch_graus=PITCH_GRAUS, log=lambda s: None)
    mostra(f"Vista completo a {v:.1f} m/s", velocidades[v])
(SAIDA / "velocidades.json").write_text(json.dumps({str(k): v for k, v in velocidades.items()}, indent=2, ensure_ascii=False))

In [ ]:
import matplotlib.pyplot as plt
vs = list(velocidades)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(vs, [velocidades[v]["taxa_conclusao"] * 100 for v in vs], "o-")
ax[0].set_xlabel("velocidade de cruzeiro (m/s)"); ax[0].set_ylabel("rotas completas (%)"); ax[0].set_ylim(0, 105)
ax[1].plot(vs, [np.median([r["tempo_s"] for r in velocidades[v]["por_rota"] if r["motivo"] == "rota_completa"] or [np.nan]) for v in vs], "o-")
ax[1].set_xlabel("velocidade de cruzeiro (m/s)"); ax[1].set_ylabel("tempo mediano da rota (s)")
plt.show()

## 9. Salvar e levar para a Jetson

In [ ]:
(SAIDA / "LEIA-ME.txt").write_text(f"""Vista v2 — commit {COMMIT}
modelos de partida: {MODELOS_DE}

faixa_final.onnx + .json + .pt   rede de faixa escolhida (melhor rodada) — use esta
placas.onnx                      detector YOLO11n ({IMG_YOLO}x{IMG_YOLO})
avaliacao_v2_modelos_anteriores.json   teto / rede / completo com os modelos de 02/11
dagger.json, dagger_N/           cada rodada (modelo + avaliação)
velocidades.json                 Vista completo em cada velocidade
pitch da câmera: {PITCH_GRAUS if PITCH_GRAUS is not None else "15 (config/camera/extrinsics.yaml)"}

Na Jetson (motor TensorRT gerado LÁ):
  /usr/src/tensorrt/bin/trtexec --onnx=faixa_final.onnx --fp16 --saveEngine=faixa.engine
  /usr/src/tensorrt/bin/trtexec --onnx=placas.onnx --fp16 --saveEngine=placas.engine
""")
for f in sorted(SAIDA.iterdir()):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f.name}" if f.is_file() else f"     dir  {f.name}/")

## Leitura dos resultados

- **teto < 100 %**: o problema não é rede — é mapa, odometria, leitura de placa ou regra. Olhe as falhas na seção 4.
- **teto alto, "c/ rede de faixa" baixo**: a rede de faixa ainda erra na faixa → mais dados / DAgger.
- **"c/ rede de faixa" alto, "completo" baixo**: o detector é o gargalo → `TREINAR_DETECTOR = True` ou olhar as placas lidas erradas.
- **`nao_largou`**: o detector não viu o verde (a rota acaba 10 s depois do verde real).
- **Velocidade**: o controle usa ganho × velocidade ≤ 2,4 (sem isso o próprio oráculo oscilava e saía da pista a 0,8 m/s).
- **Placa de rua curta**: com a câmera a 15°, a placa sai do campo de visão a menos de ~0,8 m; nas ruas de 1 m do centro ela só é vista de longe. Experimento: `PITCH_GRAUS = 5` (gera dados e treina para esse pitch) — mudar a montagem real é decisão da equipe.
- **Nunca treinar na Jetson** — só rodar o TensorRT lá.